In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical
from skopt.utils import use_named_args
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict

# 1. 加载数据
print("正在加载数据...")
df_original = pd.read_excel('../../data/quyi_data_x_names.xlsx')
print(f"原始数据集大小: {df_original.shape}")

# 2. 定义目标变量
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']

# 3. 分离特征和目标
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 4. 分割原始数据为训练集和测试集
# Paper protocol: preserve furnace order; first 900 records train, next 500 test.
if len(X_original) != 1400 or len(y_original) != 1400:
    raise ValueError('Expected 1,400 ordered furnace records (900 train + 500 test).')
X_train, X_test = X_original.iloc[:900].copy(), X_original.iloc[900:].copy()
y_train, y_test = y_original.iloc[:900].copy(), y_original.iloc[900:].copy()

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 5. 数据标准化
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# 6. 定义搜索空间 - 基于RF.ipynb中表现良好的参数范围扩展
dimensions = [
    Integer(100, 500, name='n_estimators'),
    Integer(10, 40, name='max_depth'),  # 扩大上限，因为RF.ipynb中max_depth=29表现良好
    Integer(2, 20, name='min_samples_split'),
    Integer(1, 10, name='min_samples_leaf'),
    Categorical(['sqrt', 'log2', None], name='max_features')
]

# Objective uses training-only validation; the untouched holdout is for final evaluation.
@use_named_args(dimensions)
def objective(n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features):
    params = {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'max_features': max_features,
        'random_state': 42,
        'n_jobs': -1
    }
    from sklearn.model_selection import TimeSeriesSplit
    splitter = TimeSeriesSplit(n_splits=5)
    fold_scores = []
    for train_idx, val_idx in splitter.split(X_train):
        fold_model = MultiOutputRegressor(RandomForestRegressor(**params))
        fold_model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
        pred = fold_model.predict(X_train.iloc[val_idx])
        fold_scores.append(np.mean([
            r2_score(y_train.iloc[val_idx, j], pred[:, j])
            for j in range(len(target_vars))
        ]))
    mean_r2_cv = float(np.mean(fold_scores))
    print(f"Ordered training CV R2: {mean_r2_cv:.4f}")
    return -mean_r2_cv

result = gp_minimize(
    objective,
    dimensions=dimensions,
    n_calls=30,
    n_random_starts=10,
    random_state=42,
    verbose=True,
    n_jobs=-1
)

# 9. 提取最佳参数
best_params = {dim.name: value for dim, value in zip(dimensions, result.x)}
print("\n最优参数:")
for param, value in best_params.items():
    print(f"{param}: {value}")

# 10. 使用最优参数创建新模型
print("\n用最优参数训练最终模型...")
best_model = MultiOutputRegressor(RandomForestRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1
))

# 11. 用原始训练数据训练最终模型
best_model.fit(X_train_scaled, y_train)
print("模型训练完成！")

# 12. 在测试集上进行预测
y_pred = best_model.predict(X_test_scaled)

# 13. 评估最终模型性能
print("\n优化后模型评估指标:")
overall_r2 = 0
overall_rmse = 0
overall_mae = 0

for idx, name in enumerate(target_vars):
    # 计算指标
    mse = mean_squared_error(y_test.iloc[:, idx], y_pred[:, idx])
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test.iloc[:, idx], y_pred[:, idx])
    r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
    
    # 累加指标值
    overall_r2 += r2
    overall_rmse += rmse
    overall_mae += mae
    
    # 打印每个目标变量的指标
    print(f"\n{name} 指标:")
    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")

# 14. 计算平均指标
overall_r2 /= len(target_vars)
overall_rmse /= len(target_vars)
overall_mae /= len(target_vars)

print("\n总体平均指标:")
print(f"平均 RMSE: {overall_rmse:.4f}")
print(f"平均 MAE: {overall_mae:.4f}")
print(f"平均 R²: {overall_r2:.4f}")

# 15. 输出贝叶斯优化的最佳结果
print("\n贝叶斯优化最佳结果:")
print(f"最佳综合R²: {-result.fun:.4f}")
print(f"优化迭代次数: {len(result.func_vals)}")

# 16. 为比较创建基准模型（使用原RF.ipynb中的参数）
print("\n训练对照模型...")
baseline_model = MultiOutputRegressor(RandomForestRegressor(
    n_estimators=500,
    max_depth=29,  # 使用RF.ipynb中的参数
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,  # 与RF.ipynb中使用的参数相同
    random_state=42,
    n_jobs=-1
))

# 用训练数据训练基准模型
baseline_model.fit(X_train_scaled, y_train)

# 在测试集上评估基准模型
y_pred_baseline = baseline_model.predict(X_test_scaled)

# 计算基准模型指标
baseline_r2 = 0
baseline_rmse = 0
for idx in range(len(target_vars)):
    r2 = r2_score(y_test.iloc[:, idx], y_pred_baseline[:, idx])
    mse = mean_squared_error(y_test.iloc[:, idx], y_pred_baseline[:, idx])
    rmse = np.sqrt(mse)
    baseline_r2 += r2
    baseline_rmse += rmse

baseline_r2 /= len(target_vars)
baseline_rmse /= len(target_vars)

print(f"\n基准模型平均R²: {baseline_r2:.4f}")
print(f"基准模型平均RMSE: {baseline_rmse:.4f}")
print(f"优化模型平均R²: {overall_r2:.4f}")
print(f"优化模型平均RMSE: {overall_rmse:.4f}")
print(f"R²改进: {(overall_r2 - baseline_r2):.4f}")
print(f"RMSE改进: {(baseline_rmse - overall_rmse):.4f}")